In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer
import os

# ----------------------------
# 1. Load embedding model
# ----------------------------
# This model converts text into dense vector representations (embeddings)
# Used for semantic search instead of keyword matching
# Output size = 384 dimensions (must match PostgreSQL vector column)
model = SentenceTransformer('all-MiniLM-L6-v2')

# ----------------------------
# 2. Create query embedding
# ----------------------------
# Input search query (natural language)
search_query = "survival"

# Convert query into embedding vector (384 dimensions)
# This will be compared against stored embeddings in PostgreSQL (pgvector)
query_embedding = model.encode(search_query).tolist()

# ----------------------------
# 3. Connect to Supabase (PostgreSQL)
# ----------------------------
try:
    # Establish connection to PostgreSQL (Supabase)
    # IMPORTANT:
    # - Credentials are loaded from environment variables (safe for GitHub)
    # - Never hardcode passwords in code
    # - pgvector extension must already be enabled in the database

    conn = psycopg2.connect(
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT", 5432),
        sslmode="require"
    )

    # Create cursor for executing SQL queries
    cur = conn.cursor()
    print(f"Connected to PostgreSQL! Searching for: '{search_query}'\n")

    # ----------------------------
    # 4. Execute Semantic Search (pgvector)
    # ----------------------------
    # PostgreSQL uses pgvector extension here
    # embedding <=> query_embedding = cosine distance
    # 1 - distance = similarity score (higher = more similar)
    cur.execute("""
        SELECT 
            title, 
            description, 
            1 - (embedding <=> %s::vector) AS similarity
        FROM movies
        ORDER BY similarity DESC
        LIMIT 5;
    """, (query_embedding,))

    # Fetch results from database
    rows = cur.fetchall()

    # ----------------------------
    # 5. Display Results
    # ----------------------------
    print(f"{'TITLE':<20} | {'SCORE':<7} | {'DESCRIPTION'}")
    print("-" * 75)

    # Loop through and print results
    for title, desc, score in rows:
        print(f"{title:<20} | {score:.4f} | {desc}")

# ----------------------------
# Error handling
# ----------------------------
except Exception as e:
    print(f"Error: {e}")

# ----------------------------
# Cleanup (always close DB connection)
# ----------------------------
finally:
    if 'cur' in locals():
        cur.close()
    if 'conn' in locals():
        conn.close()

    print("\nPostgreSQL connection closed.")